# **Fetch the model repository**
With Datasets and Helpers.py added

In [ ]:
!git clone -b SegGPT https://github.com/excellencior/Thesis.git
%cd Thesis
!cd SegGPT/SegGPT_inference && wget https://huggingface.co/BAAI/SegGPT/resolve/main/seggpt_vit_large.pth
!pip install -r SegGPT/SegGPT_inference/requirements.txt

# *Check Installation*

In [ ]:
!pip show torch
!pip show timm
!pip show fairscale
!nvcc --version

# **Helper function change**
Due to torch version mismatch (no import torch._six after v1.8.1)
Import moved to (Collections.abc) \
Included in **"Thesis"** repo

In [ ]:
!cp helpers.py /usr/local/lib/python3.10/dist-packages/timm/models/layers/helpers.py

# **Dataset Download**
Doesnot require execution -> If using the "My REPO"

In [ ]:
!wget 'https://huggingface.co/datasets/abturjo/dengue_test_data/resolve/main/tiles_JPG_50p.zip' # 50% overlap between the adjacent slices
!unzip tiles_JPG_50p.zip

# **Few Shot**

## Imports

In [ ]:
import os
import subprocess
import random
import numpy as np
from PIL import Image
import csv

In [ ]:
directory = "/content/tiles_JPG_50p"

# Loop through each file in the directory
for filename in os.listdir(directory):
    if filename.endswith(".jpg"):
        # Replace the specified characters in the filename
        new_filename = filename.replace("(", "-").replace(")", "-").replace(",", "-").replace(".", "-", filename.count(".") - 1)
        # Rename the file
        os.rename(os.path.join(directory, filename), os.path.join(directory, new_filename))

print("Renaming complete.")

Renaming complete.


In [ ]:
ckpt_path = "SegGPT/SegGPT_inference/seggpt_vit_large.pth"
model = "seggpt_vit_large_patch16_input896x448"

base_output_dir = "/content/predict"
os.makedirs(base_output_dir, exist_ok=True)

train_dir = os.path.join("/content/tiles_JPG_50p")
test_dir = os.path.join(base_output_dir)

# Gather all train image paths and their corresponding masks
train_images = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if "train_tile" in f and not "mask" in f])
train_masks = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if "train_mask_tile" in f])

# # Combine train images and masks, shuffle them together, then separate
# train_pairs = list(zip(train_images_all, train_masks_all))
# random.shuffle(train_pairs)
# train_images, train_masks = zip(*train_pairs[:10])

# Gather all test image paths and shuffle them
test_images = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir) if "tile" in f and not "mask" in f])
random.shuffle(test_images)

for i, test_image in enumerate(test_images, 1):
    print(f"Processing: {test_image}")
    command = f"""
    python SegGPT/SegGPT_inference/seggpt_inference.py \
    --ckpt_path {ckpt_path} \
    --model {model} \
    --input_image {test_image} \
    --prompt_image {' '.join(train_images)} \
    --prompt_target {' '.join(train_masks)} \
    --seg_type 'semantic' \
    --output_dir {base_output_dir}
    """

    try:
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(result.stdout.decode('utf-8'))
        print(result.stderr.decode('utf-8'))
        print(f"{i} inference completed, results saved to {base_output_dir}\n\n")
    except subprocess.CalledProcessError as e:
        print(f"Error during inference of {test_image}: {e.stderr.decode('utf-8')}")

print("Inference process completed.")

Processing: /content/Thesis/Data/RUN/Building/test/tile_-238191.88566-2627101.0689700004-_-13312-13312-_flower_pot.jpg
Model loaded.
Finished.


1 inference completed, results saved to /content/Thesis/Data/RUN/Building/test_result


Processing: /content/Thesis/Data/RUN/Building/test/tile_-238003.79734000002-2627289.1572900005-_-6144-6144-_flower_pot_polythene.jpg
Model loaded.
Finished.


2 inference completed, results saved to /content/Thesis/Data/RUN/Building/test_result


Processing: /content/Thesis/Data/RUN/Building/test/tile_-238030.66710000002-2627316.0270500006-_-7168-5120-_flower_pot_polythene_tyres.jpg
Model loaded.
Finished.


3 inference completed, results saved to /content/Thesis/Data/RUN/Building/test_result


Processing: /content/Thesis/Data/RUN/Building/test/tile_-238111.27638000002-2627396.6363300006-_-10240-2048-_flower_pot_open_tank_tyres.jpg
Model loaded.
Finished.


4 inference completed, results saved to /content/Thesis/Data/RUN/Building/test_result


Processing: /

# **Save the result**

In [ ]:
!zip -r "test_result_seggpt.zip" "/content/test_result"